In [3]:
import pandas as pd
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from unidecode import unidecode
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import zscore
from collections import defaultdict
from sklearn.decomposition import PCA
import phik

pds = pandas series  
df = DataFrame

# Preparando o Data Frame

In [4]:
abas  = pd.ExcelFile('WDI_EXCEL/WDIEXCEL.xlsx').sheet_names
abas


['Data', 'Country', 'Series', 'country-series', 'series-time', 'footnote']

In [5]:
metadata = pd.read_excel('WDI_EXCEL/WDIEXCEL.xlsx', sheet_name=abas[2])
metadata.head()

,Series Code,Topic,Indicator Name,Short definition,Long definition,Unit of measure,Periodicity,Base Period,Other notes,Aggregation method,Limitations and exceptions,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,Related source links,Other web links,Related indicators,License Type
0,AG.CON.FERT.PT.ZS,Environment: Agricultural production,Fertilizer consumption (% of fertilizer produc...,NaN,Fertilizer consumption measures the quantity o...,NaN,Annual,NaN,The world and regional aggregate series do not...,Weighted average,The FAO has revised the time series for fertil...,NaN,NaN,"Food and Agriculture Organization, electronic ...",Fertilizer consumption measures the quantity o...,"Factors such as the green revolution, has led ...",NaN,NaN,NaN,CC BY-4.0
1,AG.CON.FERT.ZS,Environment: Agricultural production,Fertilizer consumption (kilograms per hectare ...,NaN,Fertilizer consumption measures the quantity o...,NaN,Annual,NaN,The world and regional aggregate series do not...,Weighted average,The FAO has revised the time series for fertil...,NaN,NaN,"Food and Agriculture Organization, electronic ...",Fertilizer consumption measures the quantity o...,"Factors such as the green revolution, has led ...",NaN,NaN,NaN,CC BY-4.0
2,AG.LND.AGRI.K2,Environment: Land use,Agricultural land (sq. km),NaN,Agricultural land refers to the share of land ...,NaN,Annual,NaN,Areas of former states are included in the suc...,Sum,The data are collected by the Food and Agricul...,NaN,NaN,"Food and Agriculture Organization, electronic ...",Agricultural land constitutes only a part of a...,Agricultural land covers more than one-third o...,NaN,NaN,NaN,CC BY-4.0
3,AG.LND.AGRI.ZS,Environment: Land use,Agricultural land (% of land area),NaN,Agricultural land refers to the share of land ...,NaN,Annual,NaN,Areas of former states are included in the suc...,Weighted average,The data are collected by the Food and Agricul...,NaN,NaN,"Food and Agriculture Organization, electronic ...",Agriculture is still a major sector in many ec...,Agricultural land covers more than one-third o...,NaN,NaN,NaN,CC BY-4.0
4,AG.LND.ARBL.HA,Environment: Land use,Arable land (hectares),NaN,Arable land (in hectares) includes land define...,NaN,Annual,NaN,NaN,NaN,The Food and Agriculture Organization (FAO) tr...,NaN,NaN,"Food and Agriculture Organization, electronic ...",Temporary fallow land refers to land left fall...,Agricultural land covers more than one-third o...,NaN,NaN,NaN,CC BY-4.0


In [6]:
metadata.columns = [col.strip().replace("_", " ") for col in metadata.columns]

In [7]:
metadata["Topic"].nunique()

88

In [8]:
lista_metadata = list(metadata["Topic"].unique())

In [9]:
metadata["Indicator Name"].nunique()

1496

In [10]:
dados = pd.read_excel('WDI_EXCEL/WDIEXCEL.xlsx', sheet_name=abas[0])
dados.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,17.488497,18.001597,18.558234,19.043572,19.586457,20.192064,20.828814,21.372164,22.100884,NaN
1,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.RU.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,6.811504,7.096003,7.406706,7.666648,8.020952,8.403358,8.718306,9.097176,9.473374,NaN
2,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.UR.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,38.152090,38.488233,38.779953,39.068462,39.445526,39.818645,40.276374,40.687817,41.211606,NaN
3,Africa Eastern and Southern,AFE,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,31.871956,33.922276,38.859598,40.223744,43.035073,44.390861,46.282371,48.127211,48.742043,NaN
4,Africa Eastern and Southern,AFE,"Access to electricity, rural (% of rural popul...",EG.ELC.ACCS.RU.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,17.672943,16.527554,24.627753,25.432092,27.061929,29.154282,31.022083,32.809138,33.760782,NaN


In [11]:
dados.columns = [col.strip().replace("_", " ") for col in dados.columns]

In [12]:
print(dados.columns)


Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968',
       '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977',
       '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986',
       '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995',
       '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004',
       '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013',
       '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022',
       '2023'],
      dtype='object')


In [13]:
dados["Indicator Name"].nunique()

1496

In [14]:
dados_long = dados.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    var_name="Year",
    value_name="Value"
)

In [15]:
dados_long["Year"] = dados_long["Year"].astype(int)

In [16]:
dados_long.dtypes

Country Name       object
Country Code       object
Indicator Name     object
Indicator Code     object
Year                int64
Value             float64
dtype: object

## Separando por paises e comaçando a olhar o data frame do Brasil

Separação feita desta maneira para uma posssivel extrapolação para demais paises futuramente

In [17]:
paises_unicos = dados_long["Country Name"].unique()

sub_dfs_por_pais = {
    pais: grupo.drop(columns=["Country Code", "Country Name"]).reset_index(drop=True)
    for pais, grupo in dados_long.groupby("Country Name")
}

brasil = sub_dfs_por_pais["Brazil"]
print("\nDados do Brasil:")
brasil.head()


Dados do Brasil:


,Indicator Name,Indicator Code,Year,Value
0,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.ZS,1960,NaN
1,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.RU.ZS,1960,NaN
2,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.UR.ZS,1960,NaN
3,Access to electricity (% of population),EG.ELC.ACCS.ZS,1960,NaN
4,"Access to electricity, rural (% of rural popul...",EG.ELC.ACCS.RU.ZS,1960,NaN


# Analise do Brasil

## Explorando e preparando o data frame para analise

In [128]:
# 1. Filtrar apenas as colunas importantes para a pivotagem
df_brasil = brasil[["Year", "Indicator Name", "Value"]].copy()

# 2. Pivotar para ficar no formato "wide" (uma coluna por indicador)
df_brasil_wide = df_brasil.pivot(
    index="Year",
    columns="Indicator Name",
    values="Value"
)

# 3. Ordenar pelas datas (anos) e exibir as primeiras linhas
df_brasil_wide.sort_index(inplace=True)
df_brasil_wide.head()

Indicator Name,ARI treatment (% of children under 5 taken to a health provider),Access to clean fuels and technologies for cooking (% of population),"Access to clean fuels and technologies for cooking, rural (% of rural population)","Access to clean fuels and technologies for cooking, urban (% of urban population)",Access to electricity (% of population),"Access to electricity, rural (% of rural population)","Access to electricity, urban (% of urban population)",Account ownership at a financial institution or with a mobile-money-service provider (% of population ages 15+),"Account ownership at a financial institution or with a mobile-money-service provider, female (% of population ages 15+)","Account ownership at a financial institution or with a mobile-money-service provider, male (% of population ages 15+)",...,Women who believe a husband is justified in beating his wife (any of five reasons) (%),Women who believe a husband is justified in beating his wife when she argues with him (%),Women who believe a husband is justified in beating his wife when she burns the food (%),Women who believe a husband is justified in beating his wife when she goes out without telling him (%),Women who believe a husband is justified in beating his wife when she neglects the children (%),Women who believe a husband is justified in beating his wife when she refuses sex with him (%),Women who were first married by age 15 (% of women ages 20-24),Women who were first married by age 18 (% of women ages 20-24),Women's share of population ages 15+ living with HIV (%),Young people (ages 15-24) newly infected with HIV
Year,,,,,,,,,,,,,,,,,,,,,
1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1963,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1964,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [129]:
pds_br_tps = df_brasil_wide.dtypes
for i ,formato in pds_br_tps.items():
    if formato != 'float64' and formato != 'int64':
        print(i, formato)

Verificando quantidade de nulos por indicador

In [130]:
# Contar valores nulos por linha
pds_nulos_por_indicador = df_brasil_wide.isna().sum(axis=0)

# Exibir o resultado
print("Quantidade de valores nulos por linha (ano):")
print(pds_nulos_por_indicador.sort_values(ascending=False))

Quantidade de valores nulos por linha (ano):
Indicator Name
Women who believe a husband is justified in beating his wife when she refuses sex with him (%)       64
Vitamin A supplementation coverage rate (% of children ages 6-59 months)                             64
Young people (ages 15-24) newly infected with HIV                                                    64
Educational attainment, at least completed post-secondary, population 25+, male (%) (cumulative)     64
Educational attainment, at least completed post-secondary, population 25+, total (%) (cumulative)    64
                                                                                                     ..
Terms of trade adjustment (constant LCU)                                                              0
Agriculture, forestry, and fishing, value added (constant 2015 US$)                                   0
Agriculture, forestry, and fishing, value added (constant LCU)                                        0
Agri

In [131]:
pds_nulos_por_indicador = pds_nulos_por_indicador.sort_values(ascending=False)
indicadroes_sem_dados = []
for indicador, valor in pds_nulos_por_indicador.items():
    if valor == 64:
        indicadroes_sem_dados.append(indicador)
print(len(indicadroes_sem_dados))

102


In [132]:
new_df_brasil_wide = df_brasil_wide.drop(columns=indicadroes_sem_dados)
new_df_brasil_wide.head()

Indicator Name,ARI treatment (% of children under 5 taken to a health provider),Access to clean fuels and technologies for cooking (% of population),"Access to clean fuels and technologies for cooking, rural (% of rural population)","Access to clean fuels and technologies for cooking, urban (% of urban population)",Access to electricity (% of population),"Access to electricity, rural (% of rural population)","Access to electricity, urban (% of urban population)",Account ownership at a financial institution or with a mobile-money-service provider (% of population ages 15+),"Account ownership at a financial institution or with a mobile-money-service provider, female (% of population ages 15+)","Account ownership at a financial institution or with a mobile-money-service provider, male (% of population ages 15+)",...,"Wage and salaried workers, female (% of female employment) (modeled ILO estimate)","Wage and salaried workers, male (% of male employment) (modeled ILO estimate)","Wage and salaried workers, total (% of total employment) (modeled ILO estimate)",Wanted fertility rate (births per woman),"Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal)",Wholesale price index (2010 = 100),Women Business and the Law Index Score (scale 1-100),Women who were first married by age 15 (% of women ages 20-24),Women who were first married by age 18 (% of women ages 20-24),Women's share of population ages 15+ living with HIV (%)
Year,,,,,,,,,,,,,,,,,,,,,
1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.768224e-13,NaN,NaN,NaN,NaN
1961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2.446007e-13,NaN,NaN,NaN,NaN
1962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3.748121e-13,NaN,NaN,NaN,NaN
1963,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,6.508432e-13,NaN,NaN,NaN,NaN
1964,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.247164e-12,NaN,NaN,NaN,NaN


In [133]:
pds_nulos_por_indicador = new_df_brasil_wide.isna().sum(axis=0)
pds_nulos_por_indicador = pds_nulos_por_indicador.sort_values(ascending=False)
indicadores_remover = []
for indicador, valor in pds_nulos_por_indicador.items():
    if valor >= 24: #Garantir ao menos 40 anos de dados
        indicadores_remover.append(indicador)
print(len(indicadores_remover))

852


In [134]:
new_df_brasil_wide = new_df_brasil_wide.drop(columns=indicadores_remover)
new_df_brasil_wide.head()

Indicator Name,Adjusted net national income (annual % growth),Adjusted net national income (constant 2015 US$),Adjusted net national income (current US$),Adjusted net national income per capita (annual % growth),Adjusted net national income per capita (constant 2015 US$),Adjusted net national income per capita (current US$),Adjusted savings: consumption of fixed capital (% of GNI),Adjusted savings: consumption of fixed capital (current US$),Adjusted savings: education expenditure (% of GNI),Adjusted savings: education expenditure (current US$),...,"Travel services (% of service imports, BoP)","Unemployment, female (% of female labor force) (national estimate)","Unemployment, male (% of male labor force) (national estimate)","Unemployment, total (% of total labor force) (national estimate)",Urban population,Urban population (% of total population),Urban population growth (annual %),"Use of IMF credit (DOD, current US$)",Wholesale price index (2010 = 100),Women Business and the Law Index Score (scale 1-100)
Year,,,,,,,,,,,,,,,,,,,,,
1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,33399157.0,46.139,NaN,NaN,1.768224e-13,NaN
1961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,35155579.0,47.122,5.125267,NaN,2.446007e-13,NaN
1962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,36971452.0,48.099,5.036272,NaN,3.748121e-13,NaN
1963,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,38852223.0,49.078,4.961925,NaN,6.508432e-13,NaN
1964,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,40792376.0,50.059,4.872991,NaN,1.247164e-12,NaN


Verificando quantidade de nulos por ano

In [135]:
new_df_brasil_wide.info()

<class 'pandas.core.frame.DataFrame'>
Index: 64 entries, 1960 to 2023
Columns: 542 entries, Adjusted net national income (annual % growth) to Women Business and the Law Index Score (scale 1-100)
dtypes: float64(542)
memory usage: 271.5 KB


In [136]:
new_df_brasil_wide.describe()

Indicator Name,Adjusted net national income (annual % growth),Adjusted net national income (constant 2015 US$),Adjusted net national income (current US$),Adjusted net national income per capita (annual % growth),Adjusted net national income per capita (constant 2015 US$),Adjusted net national income per capita (current US$),Adjusted savings: consumption of fixed capital (% of GNI),Adjusted savings: consumption of fixed capital (current US$),Adjusted savings: education expenditure (% of GNI),Adjusted savings: education expenditure (current US$),...,"Travel services (% of service imports, BoP)","Unemployment, female (% of female labor force) (national estimate)","Unemployment, male (% of male labor force) (national estimate)","Unemployment, total (% of total labor force) (national estimate)",Urban population,Urban population (% of total population),Urban population growth (annual %),"Use of IMF credit (DOD, current US$)",Wholesale price index (2010 = 100),Women Business and the Law Index Score (scale 1-100)
count,51.000000,5.200000e+01,5.200000e+01,51.000000,52.000000,52.000000,52.000000,5.200000e+01,52.000000,5.200000e+01,...,49.000000,43.000000,43.000000,45.000000,6.400000e+01,64.000000,63.000000,5.400000e+01,6.200000e+01,54.000000
mean,3.010929,9.162268e+11,6.681062e+11,1.425942,5590.578090,3682.081997,14.456639,1.461168e+11,4.417439,4.287812e+10,...,18.161931,9.439349,6.268907,7.394556,1.126034e+08,71.981609,2.720236,4.433198e+09,4.324183e+01,57.060185
std,6.368678,3.304046e+11,5.915988e+11,6.130710,916.290481,2801.135806,3.945016,1.645337e+11,0.982101,4.571928e+10,...,7.024541,4.441909,2.654870,3.469044,4.910424e+07,12.931267,1.371139,6.723700e+09,6.340927e+01,22.270061
min,-14.620223,3.291109e+11,3.782767e+10,-16.131683,3450.680697,396.617704,9.018367,3.806293e+09,2.800000,1.501929e+09,...,5.898268,1.900000,1.930000,1.920000,3.339916e+07,46.139000,0.632380,0.000000e+00,1.768224e-13,28.125000
25%,-0.211241,6.445987e+11,1.925976e+11,-1.653632,4987.951719,1490.704292,11.792705,2.432008e+10,3.599282,7.718184e+09,...,13.296869,4.860000,4.056000,4.040000,6.739364e+07,61.506000,1.348287,8.609384e+07,1.767673e-11,33.125000
50%,3.097826,9.307267e+11,4.434869e+11,1.276949,5542.382954,2537.679504,13.756643,7.698704e+10,4.264303,2.063184e+10,...,18.475186,10.292000,6.008000,7.578000,1.148571e+08,75.067000,2.671691,2.533385e+09,1.448044e-03,50.625000
75%,5.882612,1.271165e+12,1.160585e+12,4.064897,6433.397369,5848.704142,15.720799,2.527057e+11,5.217500,8.456528e+10,...,22.862175,13.586500,7.900000,10.150000,1.578650e+08,83.523250,3.922914,4.443875e+09,7.878079e+01,81.875000
max,22.522958,1.435186e+12,2.003637e+12,20.438679,7230.947942,10260.080844,23.062757,5.187513e+11,6.281311,1.399386e+11,...,34.497130,16.410000,11.784000,13.697000,1.853562e+08,87.788000,5.125267,2.885033e+10,2.752110e+02,85.000000


In [137]:
# Calcular a matriz de correlação para new_df_brasil_wide
corr_matrix = new_df_brasil_wide.corr().abs()

# Remover a diagonal principal (correlação de uma variável com ela mesma)
np.fill_diagonal(corr_matrix.values, 0)

# Para cada variável, encontrar a maior correlação com outra variável
max_corr = corr_matrix.max()

# Ordenar as variáveis por valor máximo de correlação (em ordem decrescente)
high_corr_vars = max_corr.sort_values(ascending=False)

# Mostrar os top 20 indicadores com correlações mais altas
print("Top 20 indicadores com correlações mais altas:")
print(high_corr_vars.head(20))


Top 20 indicadores com correlações mais altas:
Indicator Name
Imports of goods and services (constant LCU)                                   1.0
Imports of goods and services (constant 2015 US$)                              1.0
Services, value added (constant LCU)                                           1.0
Manufacturing, value added (constant LCU)                                      1.0
Manufacturing, value added (constant 2015 US$)                                 1.0
GNI (constant LCU)                                                             1.0
Final consumption expenditure (% of GDP)                                       1.0
GNI (constant 2015 US$)                                                        1.0
Gross capital formation (constant 2015 US$)                                    1.0
Gross capital formation (constant LCU)                                         1.0
Gross domestic savings (% of GDP)                                              1.0
Services, value added (co

In [138]:
new_df_brasil_wide.columns

Index(['Adjusted net national income (annual % growth)',
       'Adjusted net national income (constant 2015 US$)',
       'Adjusted net national income (current US$)',
       'Adjusted net national income per capita (annual % growth)',
       'Adjusted net national income per capita (constant 2015 US$)',
       'Adjusted net national income per capita (current US$)',
       'Adjusted savings: consumption of fixed capital (% of GNI)',
       'Adjusted savings: consumption of fixed capital (current US$)',
       'Adjusted savings: education expenditure (% of GNI)',
       'Adjusted savings: education expenditure (current US$)',
       ...
       'Travel services (% of service imports, BoP)',
       'Unemployment, female (% of female labor force) (national estimate)',
       'Unemployment, male (% of male labor force) (national estimate)',
       'Unemployment, total (% of total labor force) (national estimate)',
       'Urban population', 'Urban population (% of total population)',
    

In [139]:
alta_correlacao = []
for col in new_df_brasil_wide.columns:
    # Encontra os pares com correlação > 0.9 para esta coluna
    correlacionados = corr_matrix[col][corr_matrix[col] > 0.9].index.tolist()
    if correlacionados:  # Se houver alguma correlação alta
        alta_correlacao.append(col)

print(f"Número de variáveis com alta correlação (>0.9): {len(alta_correlacao)}")

Número de variáveis com alta correlação (>0.9): 428


In [140]:
new_df_brasil_wide.drop(columns=alta_correlacao, inplace=True)
new_df_brasil_wide.head()

Indicator Name,Adjusted savings: mineral depletion (current US$),Adjusted savings: net forest depletion (% of GNI),Adjusted savings: net forest depletion (current US$),Adjusted savings: net national savings (% of GNI),Adjusted savings: net national savings (current US$),Agricultural raw materials imports (% of merchandise imports),"Agriculture, forestry, and fishing, value added (% of GDP)","Agriculture, forestry, and fishing, value added (annual % growth)",Arms exports (SIPRI trend indicator values),Arms imports (SIPRI trend indicator values),...,Sex ratio at birth (male births per female births),"Short-term debt (% of exports of goods, services and primary income)",Short-term debt (% of total external debt),Short-term debt (% of total reserves),Surface area (sq. km),Terms of trade adjustment (constant LCU),Total debt service (% of GNI),"Total debt service (% of exports of goods, services and primary income)",Total reserves in months of imports,"Use of IMF credit (DOD, current US$)"
Year,,,,,,,,,,,,,,,,,,,,,
1960,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,2000000.0,236000000.0,...,1.045,NaN,NaN,NaN,NaN,7.572936e+09,NaN,NaN,NaN,NaN
1961,NaN,NaN,NaN,NaN,NaN,NaN,0.0,7.6,12000000.0,212000000.0,...,1.045,NaN,NaN,NaN,8515770.0,6.225752e+09,NaN,NaN,NaN,NaN
1962,NaN,NaN,NaN,NaN,NaN,2.922021,0.0,0.5,NaN,62000000.0,...,1.045,NaN,NaN,NaN,8515770.0,2.218923e+09,NaN,NaN,NaN,NaN
1963,NaN,NaN,NaN,NaN,NaN,2.949165,0.0,1.0,NaN,127000000.0,...,1.045,NaN,NaN,NaN,8515770.0,3.096473e+09,NaN,NaN,NaN,NaN
1964,NaN,NaN,NaN,NaN,NaN,2.217361,0.0,1.3,NaN,25000000.0,...,1.046,NaN,NaN,NaN,8515770.0,5.057024e+09,NaN,NaN,NaN,NaN


In [141]:
corr_matrix_phik = new_df_brasil_wide.phik_matrix().abs()

# Remover a diagonal principal (correlação de uma variável com ela mesma)
np.fill_diagonal(corr_matrix_phik.values, 0)

interval columns not set, guessing: ['Adjusted savings: mineral depletion (current US$)', 'Adjusted savings: net forest depletion (% of GNI)', 'Adjusted savings: net forest depletion (current US$)', 'Adjusted savings: net national savings (% of GNI)', 'Adjusted savings: net national savings (current US$)', 'Agricultural raw materials imports (% of merchandise imports)', 'Agriculture, forestry, and fishing, value added (% of GDP)', 'Agriculture, forestry, and fishing, value added (annual % growth)', 'Arms exports (SIPRI trend indicator values)', 'Arms imports (SIPRI trend indicator values)', 'Average precipitation in depth (mm per year)', 'Broad money (% of GDP)', 'Broad money to total reserves ratio', 'Capture fisheries production (metric tons)', 'Carbon intensity of GDP (kg CO2e per constant 2015 US$ of GDP)', 'Changes in inventories (current LCU)', 'Changes in inventories (current US$)', 'Claims on central government (annual growth as % of broad money)', 'Claims on central government

/home/pedro/Documentos/025.1/SME0852/.venv/lib/python3.12/site-packages/phik/data_quality.py:72: UserWarning: Not enough unique value for variable Adjusted savings: net forest depletion (% of GNI) for analysis 1. Dropping this column
  warnings.warn(
/home/pedro/Documentos/025.1/SME0852/.venv/lib/python3.12/site-packages/phik/data_quality.py:72: UserWarning: Not enough unique value for variable Adjusted savings: net forest depletion (current US$) for analysis 1. Dropping this column
  warnings.warn(
/home/pedro/Documentos/025.1/SME0852/.venv/lib/python3.12/site-packages/phik/data_quality.py:72: UserWarning: Not enough unique value for variable Land area (sq. km) for analysis 1. Dropping this column
  warnings.warn(
/home/pedro/Documentos/025.1/SME0852/.venv/lib/python3.12/site-packages/phik/data_quality.py:72: UserWarning: Not enough unique value for variable Lower secondary school starting age (years) for analysis 1. Dropping this column
  warnings.warn(
/home/pedro/Documentos/025.1/S

In [142]:
# Calcula a matriz de correlação Phi-K e converte para valores absolutos
corr_matrix_phik = new_df_brasil_wide.phik_matrix().abs()

# Remove a diagonal principal (correlação de uma variável com ela mesma)
np.fill_diagonal(corr_matrix_phik.values, 0)

# Lista para armazenar variáveis com alta correlação
alta_correlacao = []
# Dicionário para armazenar quais variáveis estão correlacionadas com cada coluna
pares_correlacionados = {}

for col in new_df_brasil_wide.columns:
    # Encontra os pares com correlação > 0.9 para esta coluna
    correlacionados = corr_matrix_phik[col][corr_matrix_phik[col] > 0.9].index.tolist()
    if correlacionados:  # Se houver alguma correlação alta
        alta_correlacao.append(col)
        pares_correlacionados[col] = correlacionados

print(f"Número de variáveis com alta correlação (>0.9): {len(alta_correlacao)}")

interval columns not set, guessing: ['Adjusted savings: mineral depletion (current US$)', 'Adjusted savings: net forest depletion (% of GNI)', 'Adjusted savings: net forest depletion (current US$)', 'Adjusted savings: net national savings (% of GNI)', 'Adjusted savings: net national savings (current US$)', 'Agricultural raw materials imports (% of merchandise imports)', 'Agriculture, forestry, and fishing, value added (% of GDP)', 'Agriculture, forestry, and fishing, value added (annual % growth)', 'Arms exports (SIPRI trend indicator values)', 'Arms imports (SIPRI trend indicator values)', 'Average precipitation in depth (mm per year)', 'Broad money (% of GDP)', 'Broad money to total reserves ratio', 'Capture fisheries production (metric tons)', 'Carbon intensity of GDP (kg CO2e per constant 2015 US$ of GDP)', 'Changes in inventories (current LCU)', 'Changes in inventories (current US$)', 'Claims on central government (annual growth as % of broad money)', 'Claims on central government

/home/pedro/Documentos/025.1/SME0852/.venv/lib/python3.12/site-packages/phik/data_quality.py:72: UserWarning: Not enough unique value for variable Adjusted savings: net forest depletion (% of GNI) for analysis 1. Dropping this column
  warnings.warn(
/home/pedro/Documentos/025.1/SME0852/.venv/lib/python3.12/site-packages/phik/data_quality.py:72: UserWarning: Not enough unique value for variable Adjusted savings: net forest depletion (current US$) for analysis 1. Dropping this column
  warnings.warn(
/home/pedro/Documentos/025.1/SME0852/.venv/lib/python3.12/site-packages/phik/data_quality.py:72: UserWarning: Not enough unique value for variable Land area (sq. km) for analysis 1. Dropping this column
  warnings.warn(
/home/pedro/Documentos/025.1/SME0852/.venv/lib/python3.12/site-packages/phik/data_quality.py:72: UserWarning: Not enough unique value for variable Lower secondary school starting age (years) for analysis 1. Dropping this column
  warnings.warn(
/home/pedro/Documentos/025.1/S

KeyError: 'Adjusted savings: net forest depletion (% of GNI)'

## Condensando as variaveis

### Topico com subtopico

In [ ]:
# Filtra indicadores da metadata que estão no df_brasil_wide
metadata_filtered = metadata[metadata["Indicator Name"].isin(new_df_brasil_wide.columns)]

# Agrupa por tópico
agrupamentos_por_topico = (
    metadata_filtered.groupby("Topic")["Indicator Name"]
    .apply(list)
    .to_dict()
)


In [ ]:
x = []
for topico in agrupamentos_por_topico.keys():
    z = []
    for item in agrupamentos_por_topico[topico]:
      z.append(item)
    x.append(len(z))
print(x)

[9, 4, 22, 7, 6, 8, 9, 4, 13, 20, 2, 14, 6, 7, 3, 4, 9, 10, 4, 18, 5, 8, 4, 8, 9, 4, 30, 3, 2, 1, 1, 11, 11, 15, 2, 9, 6, 8, 3, 8, 1, 4, 2, 17, 6, 58, 2, 4, 4, 3, 23, 23, 1, 2, 5, 7, 3, 3, 5]


In [ ]:
# 3. Cria um dicionário de DataFrames por tópico
dfs_por_topico = {}

for topico, indicadores in agrupamentos_por_topico.items():
    # Garante que os indicadores existem no DataFrame
    indicadores_validos = [ind for ind in indicadores if ind in new_df_brasil_wide.columns]
    
    if indicadores_validos:
        df_topico = new_df_brasil_wide[indicadores_validos].copy()
        dfs_por_topico[topico] = df_topico


In [ ]:
dfs_por_topico["Health: Mortality"].head()

Indicator Name,Number of infant deaths,Number of under-five deaths,Number of neonatal deaths,"Mortality rate, under-5 (per 1,000 live births)","Mortality rate, under-5, female (per 1,000 live births)","Mortality rate, under-5, male (per 1,000 live births)","Mortality rate, neonatal (per 1,000 live births)","Mortality rate, adult, female (per 1,000 female adults)","Mortality rate, adult, male (per 1,000 male adults)","Mortality rate, infant, female (per 1,000 live births)","Mortality rate, infant (per 1,000 live births)","Mortality rate, infant, male (per 1,000 live births)","Life expectancy at birth, female (years)","Life expectancy at birth, total (years)","Life expectancy at birth, male (years)","Survival to age 65, female (% of cohort)","Survival to age 65, male (% of cohort)"
Year,,,,,,,,,,,,,,,,,
1960,395422.0,519510.0,NaN,169.1,155.7,182.0,NaN,278.299,354.358,114.0,127.4,140.1,55.241,52.660,50.281,52.773369,43.797373
1961,394017.0,516918.0,NaN,164.6,151.2,177.5,NaN,274.079,349.877,111.0,124.2,137.0,55.782,53.183,50.788,53.391814,44.345563
1962,391852.0,513552.0,NaN,160.4,147.0,173.1,NaN,270.721,344.128,108.1,121.3,133.9,56.268,53.710,51.346,54.011894,44.875258
1963,388414.0,508777.0,NaN,156.2,143.0,168.8,49.9,266.477,339.704,105.4,118.4,130.8,56.776,54.209,51.835,54.675386,45.444753
1964,383551.0,502497.0,167269.0,152.4,139.4,164.8,49.3,263.135,335.834,102.8,115.7,128.0,57.223,54.648,52.265,55.282381,45.940763


In [ ]:
dfs_por_topico["Health: Mortality"].describe()

Indicator Name,Number of infant deaths,Number of under-five deaths,Number of neonatal deaths,"Mortality rate, under-5 (per 1,000 live births)","Mortality rate, under-5, female (per 1,000 live births)","Mortality rate, under-5, male (per 1,000 live births)","Mortality rate, neonatal (per 1,000 live births)","Mortality rate, adult, female (per 1,000 female adults)","Mortality rate, adult, male (per 1,000 male adults)","Mortality rate, infant, female (per 1,000 live births)","Mortality rate, infant (per 1,000 live births)","Mortality rate, infant, male (per 1,000 live births)","Life expectancy at birth, female (years)","Life expectancy at birth, total (years)","Life expectancy at birth, male (years)","Survival to age 65, female (% of cohort)","Survival to age 65, male (% of cohort)"
count,63.000000,63.000000,59.000000,63.000000,63.000000,63.000000,60.000000,63.000000,63.000000,63.000000,63.000000,63.000000,63.000000,63.000000,63.000000,63.000000,63.000000
mean,195819.920635,244925.666667,92150.830508,70.544444,63.896825,76.880952,26.326667,163.767778,249.124540,49.976190,56.276190,62.273016,68.551349,65.599762,62.775032,71.959371,60.659164
std,127413.436332,169316.250471,51142.068520,49.931407,45.801651,53.860602,13.892272,62.939475,51.712449,32.969021,37.086036,41.003046,7.318858,6.960400,6.572837,10.857894,9.437876
min,34276.000000,38575.000000,23338.000000,14.000000,12.400000,15.500000,8.600000,75.626000,161.413000,11.100000,12.500000,13.900000,55.241000,52.660000,50.281000,52.773369,43.797373
25%,63480.500000,71511.000000,38429.000000,22.500000,19.800000,25.000000,12.550000,107.447500,211.747500,17.600000,20.000000,22.300000,62.020000,59.572500,57.258000,62.251292,52.929204
50%,186096.000000,223327.000000,89754.000000,60.200000,54.200000,65.900000,25.000000,153.000000,239.996000,45.000000,50.400000,55.600000,69.335000,66.310000,63.442000,73.479743,61.116517
75%,317342.500000,405035.500000,144650.000000,113.800000,103.250000,123.900000,38.475000,219.752500,286.671000,78.650000,88.750000,98.400000,75.772500,72.201000,68.750500,82.161108,69.047276
max,395422.000000,519510.000000,167269.000000,169.100000,155.700000,182.000000,49.900000,278.299000,354.358000,114.000000,127.400000,140.100000,78.469000,75.338000,72.203000,86.473098,75.501009


In [ ]:
diagnostico_topico = []

for topico, df in dfs_por_topico.items():
    df_clean = df.dropna(axis=1, how='all').dropna()  # remove colunas completamente nulas e linhas com NA

    if df_clean.empty:
        continue

    # Padroniza os dados
    scaler = StandardScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df_clean), columns=df_clean.columns)

    # Contagem total de outliers (z-score > 3)
    outliers_mask = df_scaled.abs() > 3
    n_outliers_total = outliers_mask.sum().sum()

    # Quantas variáveis têm ao menos 1 outlier
    n_vars_outlier = (outliers_mask.sum() > 0).sum()
    pct_vars_outlier = round((n_vars_outlier / len(df_clean)), 2)

    # Correlações
    try:
        corr_matrix = df_scaled.corr().abs()
        upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        n_corr_alta = (upper_tri > 0.9).sum().sum()
        n_vars = df_scaled.shape[1]
        if n_vars <= 1:
            per_cent_corr = 0.0
        else:
            total_possible = n_vars * (n_vars - 1) / 2
            per_cent_corr = round((n_corr_alta / total_possible) * 100, 2)
    except:
        per_cent_corr = None

    diagnostico_topico.append({
        'Topico': topico,
        'N_Variáveis': df.shape[1],
        'N_Observações': df.shape[0],
        '% Missing': round(df.isnull().mean().mean() * 100, 2),
        '% correlações > 0.9': per_cent_corr,
        'N_Outliers (z > 3)': int(n_outliers_total),
        '% Variáveis com Outliers': pct_vars_outlier
    })

df_diag = pd.DataFrame(diagnostico_topico).sort_values('Topico', ascending=True).reset_index(drop=True)
df_diag

,Topico,N_Variáveis,N_Observações,% Missing,% correlações > 0.9,N_Outliers (z > 3),% Variáveis com Outliers
0,Economic Policy & Debt: Balance of payments: C...,9,64,19.97,5.56,5,0.08
1,Economic Policy & Debt: Balance of payments: C...,4,64,23.44,0.00,2,0.04
2,Economic Policy & Debt: Balance of payments: C...,22,64,23.44,25.54,2,0.04
3,Economic Policy & Debt: Balance of payments: C...,7,64,24.33,9.52,2,0.04
4,Economic Policy & Debt: Balance of payments: R...,6,64,7.29,20.00,1,0.02
5,Economic Policy & Debt: External debt: Debt ou...,8,64,15.62,53.57,2,0.02
6,Economic Policy & Debt: External debt: Debt ra...,9,64,18.23,0.00,3,0.06
7,Economic Policy & Debt: External debt: Debt se...,4,64,19.92,0.00,3,0.07
8,Economic Policy & Debt: External debt: Net flows,13,64,17.79,2.56,14,0.28
9,Economic Policy & Debt: National accounts: Adj...,20,64,20.08,11.05,7,0.15


### Topicos macros

In [ ]:
dfs_por_macrotema = defaultdict(list)

for topico, df in dfs_por_topico.items():
    macrotema = topico.split(':')[0].strip()
    dfs_por_macrotema[macrotema].append(df)


In [ ]:
dfs_macrotema_unificado = {}

for macrotema, lista_dfs in dfs_por_macrotema.items():
    df_merged = pd.concat(lista_dfs, axis=1)
    # Remove colunas duplicadas (caso haja) e mantém ordem
    df_merged = df_merged.loc[:, ~df_merged.columns.duplicated()]
    dfs_macrotema_unificado[macrotema] = df_merged

In [ ]:
diagnostico_macrotema = []

for tema, df in dfs_macrotema_unificado.items():
    df_clean = df.dropna(axis=1, how='all').dropna()  # remove colunas completamente nulas e linhas com NA

    if df_clean.empty:
        continue

    # Padroniza os dados
    scaler = StandardScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df_clean), columns=df_clean.columns)

    # Contagem total de outliers (z-score > 3)
    outliers_mask = df_scaled.abs() > 3
    n_outliers_total = outliers_mask.sum().sum()

    # Quantas variáveis têm ao menos 1 outlier
    n_vars_outlier = (outliers_mask.sum() > 0).sum()
    pct_vars_outlier = round((n_vars_outlier / len(df_clean)), 2)

    # Correlações
    try:
        corr_matrix = df_scaled.corr().abs()
        upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        n_corr_alta = (upper_tri > 0.9).sum().sum()
        n_vars = df_scaled.shape[1]
        if n_vars <= 1:
            per_cent__corr = 0.0  # Ou np.nan, dependendo do que quiser representar
        else:
            total_possible = n_vars * (n_vars - 1) / 2
            per_cent__corr = round((n_corr_alta / total_possible) * 100, 2)

    except:
        per_cent__corr = None

    diagnostico_macrotema.append({
        'Macrotema': tema,
        'N_Variáveis': df.shape[1],
        'N_Observações': df.shape[0],
        '% Missing': round(df.isnull().mean().mean() * 100, 2),
        '% correlações > 0.9': per_cent__corr,
        'N_Outliers (z > 3)': int(n_outliers_total),
        '% Variáveis com Outliers': pct_vars_outlier
    })

df_diag_macro = pd.DataFrame(diagnostico_macrotema).sort_values('N_Variáveis', ascending=False)
df_diag_macro.reset_index(drop=True, inplace=True)
df_diag_macro


,Macrotema,N_Variáveis,N_Observações,% Missing,% correlações > 0.9,N_Outliers (z > 3),% Variáveis com Outliers
0,Economic Policy & Debt,247,64,11.74,11.59,32,1.45
1,Health,85,64,1.29,54.57,0,0.00
2,Environment,54,64,12.07,12.72,12,0.23
3,Private Sector & Trade,49,64,9.47,5.10,5,0.12
4,Financial Sector,24,64,10.61,11.96,15,0.32
5,Social Protection & Labor,13,64,30.17,8.97,2,0.06
6,Infrastructure,11,64,21.16,40.00,0,0.00
7,Education,6,64,20.05,6.67,0,0.00
8,Public Sector,5,64,4.38,0.00,1,0.02
9,Trade,5,64,34.38,60.00,0,0.00


In [ ]:
dfs_macrotema_unificado['Health'].head()

Indicator Name,"Immunization, DPT (% of children ages 12-23 months)","Immunization, measles (% of children ages 12-23 months)",Number of infant deaths,Number of under-five deaths,Number of neonatal deaths,"Mortality rate, under-5 (per 1,000 live births)","Mortality rate, under-5, female (per 1,000 live births)","Mortality rate, under-5, male (per 1,000 live births)","Mortality rate, neonatal (per 1,000 live births)","Mortality rate, adult, female (per 1,000 female adults)",...,"Population ages 80 and above, female (% of female population)","Population ages 80 and above, male (% of male population)",Sex ratio at birth (male births per female births),"Population, total","Population, female","Population, female (% of total population)","Population, male","Population, male (% of total population)","Adolescent fertility rate (births per 1,000 women ages 15-19)","Fertility rate, total (births per woman)"
Year,,,,,,,,,,,,,,,,,,,,,
1960,NaN,NaN,395422.0,519510.0,NaN,169.1,155.7,182.0,NaN,278.299,...,0.271983,0.187578,1.045,72388126.0,36100035.0,49.870105,36288091.0,50.129895,91.302,6.061
1961,NaN,NaN,394017.0,516918.0,NaN,164.6,151.2,177.5,NaN,274.079,...,0.270764,0.188337,1.045,74605447.0,37220571.0,49.889884,37384876.0,50.110116,90.889,6.044
1962,NaN,NaN,391852.0,513552.0,NaN,160.4,147.0,173.1,NaN,270.721,...,0.270551,0.189714,1.045,76865323.0,38362620.0,49.908877,38502703.0,50.091123,90.139,5.995
1963,NaN,NaN,388414.0,508777.0,NaN,156.2,143.0,168.8,49.9,266.477,...,0.271003,0.191623,1.045,79164235.0,39524520.0,49.927244,39639715.0,50.072756,89.118,5.929
1964,NaN,NaN,383551.0,502497.0,167269.0,152.4,139.4,164.8,49.3,263.135,...,0.272911,0.194715,1.046,81488595.0,40699671.0,49.945236,40788924.0,50.054764,87.591,5.818


In [ ]:
slcts = ['Trade', 'Infrastructure', 'Health']

for tema in slcts:
    data = dfs_macrotema_unificado[tema].reset_index(drop=True)

    # Substituindo os valores NaN pela média de cada coluna
    data = data.fillna(data.mean())

    # Normalizando os dados
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)

    # Aplicando PCA e encontrando o número mínimo de componentes que explicam >= 90% da variância
    for n in range(1, min(data_scaled.shape[1], 6)):  # até 5 ou até o número de colunas
        pca = PCA(n_components=n)
        pca.fit(data_scaled)
        if sum(pca.explained_variance_ratio_) >= 0.9:
            break

    print(f"\n{tema} — Variância explicada: {pca.explained_variance_ratio_}")

    # Obtendo os scores do PCA
    pca_scores = pca.transform(data_scaled)

    # Substituindo o DataFrame original pelas componentes principais
    dfs_macrotema_unificado[tema] = pd.DataFrame(
        pca_scores,
        index=dfs_macrotema_unificado[tema].index,
        columns=[f'PC{i+1}' for i in range(pca.n_components_)]
    )



Trade — Variância explicada: [0.92460074]

Infrastructure — Variância explicada: [0.76050708 0.12872628 0.0484004 ]

Health — Variância explicada: [0.82013349 0.11066274]


In [ ]:
dfs_macrotema_unificado['Health'].head()

,PC1,PC2
Year,,
1960,11.752438,-4.609571
1961,11.764746,-4.479053
1962,11.556293,-4.335871
1963,11.531033,-4.238402
1964,11.483789,-3.951403


In [ ]:
dfs_macrotema_unificado['Trade'].head()

,PC1
Year,
1960,-2.620427e-16
1961,-2.620427e-16
1962,-2.620427e-16
1963,-2.620427e-16
1964,-2.620427e-16


In [ ]:
# Lista de tópicos a verificar
topicos_a_verificar_dps = ['Trade', 'Infrastructure', 'Health', 'Public Sector', 'Education']

# Filtra o DataFrame removendo linhas onde a coluna 'Topico' contém qualquer uma das palavras em 'topicos_a_verificar_dps'
df_diag_filtrado = df_diag[~df_diag['Topico'].str.contains('|'.join(topicos_a_verificar_dps), case=False, na=False)].reset_index(drop=True)

df_diag_filtrado


,Topico,N_Variáveis,N_Observações,% Missing,% correlações > 0.9,N_Outliers (z > 3),% Variáveis com Outliers
0,Economic Policy & Debt: Balance of payments: C...,9,64,19.97,5.56,5,0.08
1,Economic Policy & Debt: Balance of payments: C...,4,64,23.44,0.00,2,0.04
2,Economic Policy & Debt: Balance of payments: C...,22,64,23.44,25.54,2,0.04
3,Economic Policy & Debt: Balance of payments: C...,7,64,24.33,9.52,2,0.04
4,Economic Policy & Debt: Balance of payments: R...,6,64,7.29,20.00,1,0.02
5,Economic Policy & Debt: External debt: Debt ou...,8,64,15.62,53.57,2,0.02
6,Economic Policy & Debt: External debt: Debt ra...,9,64,18.23,0.00,3,0.06
7,Economic Policy & Debt: External debt: Debt se...,4,64,19.92,0.00,3,0.07
8,Economic Policy & Debt: External debt: Net flows,13,64,17.79,2.56,14,0.28
9,Economic Policy & Debt: National accounts: Adj...,20,64,20.08,11.05,7,0.15


In [ ]:
topicos_alta_correlacao = df_diag_filtrado[df_diag_filtrado['% correlações > 0.9'] > 15]['Topico'].tolist()

In [ ]:
topicos_alta_correlacao

['Economic Policy & Debt: Balance of payments: Current account: Goods, services & income',
 'Economic Policy & Debt: Balance of payments: Reserves & other items',
 'Economic Policy & Debt: External debt: Debt outstanding',
 'Economic Policy & Debt: National accounts: Atlas GNI & GNI per capita',
 'Economic Policy & Debt: National accounts: Local currency at constant prices: Aggregate indicators',
 'Economic Policy & Debt: National accounts: Local currency at constant prices: Expenditure on GDP',
 'Economic Policy & Debt: National accounts: Local currency at constant prices: Other items',
 'Economic Policy & Debt: National accounts: Local currency at constant prices: Value added',
 'Economic Policy & Debt: National accounts: Local currency at current prices: Aggregate indicators',
 'Economic Policy & Debt: National accounts: Local currency at current prices: Expenditure on GDP',
 'Economic Policy & Debt: National accounts: Local currency at current prices: Value added',
 'Economic Polic

In [ ]:
for tema in topicos_alta_correlacao:
    data = df_diag[tema].reset_index(drop=True)

    # Substituindo os valores NaN pela média de cada coluna
    data = data.fillna(data.mean())

    # Normalizando os dados
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)

    # Aplicando PCA e encontrando o número mínimo de componentes que explicam >= 90% da variância
    for n in range(1, min(data_scaled.shape[1], 6)):  # até 5 ou até o número de colunas
        pca = PCA(n_components=n)
        pca.fit(data_scaled)
        if sum(pca.explained_variance_ratio_) >= 0.9:
            break

    print(f"\n{tema} — Variância explicada: {pca.explained_variance_ratio_}")

    # Obtendo os scores do PCA
    pca_scores = pca.transform(data_scaled)

    # Substituindo o DataFrame original pelas componentes principais
    df_diag_filtrado[tema] = pd.DataFrame(
        pca_scores,
        index=dfs_macrotema_unificado[tema].index,
        columns=[f'PC{i+1}' for i in range(pca.n_components_)]
    )


KeyError: 'Economic Policy & Debt: Balance of payments: Current account: Goods, services & income'